In [ ]:
!pip install -U deepeval openai pandas pydantic PyPDF2
!pip install langchain-google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 858.6/858.6 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 87.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 27.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.3/228.3 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 2.3 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.4
    Uninstalling pydantic_core-2.4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **gpt-4o**

In [ ]:
import json
import glob
import os
import time
import pandas as pd
from google.colab import userdata

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

os.environ["OPENAI_API_KEY"] = userdata.get('AgentEval')

def load_interview_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    cv_string = json.dumps(data.get('candidate_cv', {}), indent=2)
    combined_input = f"JOB DESCRIPTION:\n{data.get('job_description', '')}\n\nCANDIDATE CV:\n{cv_string}"

    transcript_str = "\n".join([f"{t['role'].upper()}: {t['content']}" for t in data.get('transcript', [])])

    scorecard = data.get('scorecard', None)
    scorecard_str = json.dumps(scorecard, indent=2) if scorecard else None

    return combined_input, transcript_str, scorecard_str

def evaluate_all_interviews_baseline(folder_path):

    prompt_compliance_metric = GEval(
        name="Prompt-Compliance Baseline",
        criteria="Score the interviewer agent's prompt compliance and conversational behavior on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    semantic_relevance_metric = GEval(
        name="Semantic Relevance Baseline",
        criteria="Score the semantic alignment of the interviewer's questions with the Job Description and Candidate CV on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    structural_consistency_metric = GEval(
        name="Structural Consistency Baseline",
        criteria="Score the structural flow and logical progression of the interview on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    evaluation_validity_metric = GEval(
        name="Evaluation Validity Baseline",
        criteria="Score the logic of the generated scorecard on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-4o"
    )

    all_results = []
    files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Starting BASELINE evaluation of {len(files)} interviews...\n")

    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"Evaluating Baseline: {file_name}...")

        system_input, transcript_output, scorecard_output = load_interview_data(file_path)

        interviewer_test_case = LLMTestCase(
            input=system_input,
            actual_output=transcript_output
        )

        prompt_compliance_metric.measure(interviewer_test_case)
        semantic_relevance_metric.measure(interviewer_test_case)
        structural_consistency_metric.measure(interviewer_test_case)

        eval_score = None
        eval_reason = "No scorecard provided in JSON."

        if scorecard_output:
            scorecard_test_case = LLMTestCase(
                input=f"{system_input}\n\nTRANSCRIPT:\n{transcript_output}",
                actual_output=scorecard_output
            )
            evaluation_validity_metric.measure(scorecard_test_case)
            eval_score = evaluation_validity_metric.score
            eval_reason = evaluation_validity_metric.reason

        print(f"--- Baseline Judgement for {file_name} ---")
        print(f"Prompt Compliance: {prompt_compliance_metric.score}")
        print(f"Semantic Relevance: {semantic_relevance_metric.score}")
        print(f"Structural Consistency: {structural_consistency_metric.score}")
        print(f"Evaluation Validity: {eval_score if eval_score is not None else 'N/A'}")
        print("-" * 40)

        all_results.append({
            "Interview_ID": file_name,
            "Baseline_Prompt_Compliance_Score": prompt_compliance_metric.score,
            "Baseline_Semantic_Relevance_Score": semantic_relevance_metric.score,
            "Baseline_Structural_Consistency_Score": structural_consistency_metric.score,
            "Baseline_Eval_Validity_Score": eval_score,
        })

        time.sleep(25)

    df = pd.DataFrame(all_results)

    print("\n========================================")
    print("FULL BASELINE EVALUATION METRICS")
    print("========================================")
    print(f"Total Interviews Processed: {len(df)}")
    print(f"Avg Baseline Prompt Compliance: {df['Baseline_Prompt_Compliance_Score'].mean():.2f}")
    print(f"Avg Baseline Semantic Relevance: {df['Baseline_Semantic_Relevance_Score'].mean():.2f}")
    print(f"Avg Baseline Structural Consistency: {df['Baseline_Structural_Consistency_Score'].mean():.2f}")

    valid_scorecards = df['Baseline_Eval_Validity_Score'].dropna()
    if not valid_scorecards.empty:
        print(f"Avg Baseline Evaluation Validity: {valid_scorecards.mean():.2f} ({len(valid_scorecards)} scorecards evaluated)")
    else:
        print("Avg Baseline Evaluation Validity: N/A (No scorecards found)")

    csv_filename = "full_agent_evaluation_report_ZERO_SHOT_BASELINE.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nDetailed baseline report saved to: {csv_filename}")

if __name__ == "__main__":
    evaluate_all_interviews_baseline("/content/drive/MyDrive/Evaluation Dataset")

Output()

Starting BASELINE evaluation of 13 interviews...

Evaluating Baseline: Interview Eval #1.json...


Output()

Output()

--- Baseline Judgement for Interview Eval #1.json ---
Prompt Compliance: 0.5633720292865572
Semantic Relevance: 0.2461737548195928
Structural Consistency: 0.3118730164036578
Evaluation Validity: N/A
----------------------------------------
Evaluating Baseline: Interview Eval #2.json...


...

Output()

Output()

Output()

Output()

--- Baseline Judgement for Interview Eval #12.json ---
Prompt Compliance: 0.5513082974339734
Semantic Relevance: 0.29977801891589145
Structural Consistency: 0.3087513180147968
Evaluation Validity: 0.40894773597956025
----------------------------------------

FULL BASELINE EVALUATION METRICS
Total Interviews Processed: 13
Avg Baseline Prompt Compliance: 0.67
Avg Baseline Semantic Relevance: 0.38
Avg Baseline Structural Consistency: 0.51
Avg Baseline Evaluation Validity: 0.34 (8 scorecards evaluated)

Detailed baseline report saved to: full_agent_evaluation_report_ZERO_SHOT_BASELINE.csv


# **gpt-5.4-mini**

In [ ]:
import json
import glob
import os
import time
import pandas as pd
from google.colab import userdata

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

os.environ["OPENAI_API_KEY"] = userdata.get('AgentEval')

def load_interview_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    cv_string = json.dumps(data.get('candidate_cv', {}), indent=2)
    combined_input = f"JOB DESCRIPTION:\n{data.get('job_description', '')}\n\nCANDIDATE CV:\n{cv_string}"

    transcript_str = "\n".join([f"{t['role'].upper()}: {t['content']}" for t in data.get('transcript', [])])

    scorecard = data.get('scorecard', None)
    scorecard_str = json.dumps(scorecard, indent=2) if scorecard else None

    return combined_input, transcript_str, scorecard_str

def evaluate_all_interviews_baseline(folder_path):

    prompt_compliance_metric = GEval(
        name="Prompt-Compliance Baseline",
        criteria="Score the interviewer agent's prompt compliance and conversational behavior on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    semantic_relevance_metric = GEval(
        name="Semantic Relevance Baseline",
        criteria="Score the semantic alignment of the interviewer's questions with the Job Description and Candidate CV on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    structural_consistency_metric = GEval(
        name="Structural Consistency Baseline",
        criteria="Score the structural flow and logical progression of the interview on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    evaluation_validity_metric = GEval(
        name="Evaluation Validity Baseline",
        criteria="Score the logic of the generated scorecard on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model="gpt-5.4-mini-2026-03-17"
    )

    all_results = []
    files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Starting BASELINE evaluation of {len(files)} interviews...\n")

    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"Evaluating Baseline: {file_name}...")

        system_input, transcript_output, scorecard_output = load_interview_data(file_path)

        interviewer_test_case = LLMTestCase(
            input=system_input,
            actual_output=transcript_output
        )

        prompt_compliance_metric.measure(interviewer_test_case)
        semantic_relevance_metric.measure(interviewer_test_case)
        structural_consistency_metric.measure(interviewer_test_case)

        eval_score = None
        eval_reason = "No scorecard provided in JSON."

        if scorecard_output:
            scorecard_test_case = LLMTestCase(
                input=f"{system_input}\n\nTRANSCRIPT:\n{transcript_output}",
                actual_output=scorecard_output
            )
            evaluation_validity_metric.measure(scorecard_test_case)
            eval_score = evaluation_validity_metric.score
            eval_reason = evaluation_validity_metric.reason

        print(f"--- Baseline Judgement for {file_name} ---")
        print(f"Prompt Compliance: {prompt_compliance_metric.score}")
        print(f"Semantic Relevance: {semantic_relevance_metric.score}")
        print(f"Structural Consistency: {structural_consistency_metric.score}")
        print(f"Evaluation Validity: {eval_score if eval_score is not None else 'N/A'}")
        print("-" * 40)

        all_results.append({
            "Interview_ID": file_name,
            "Baseline_Prompt_Compliance_Score": prompt_compliance_metric.score,
            "Baseline_Semantic_Relevance_Score": semantic_relevance_metric.score,
            "Baseline_Structural_Consistency_Score": structural_consistency_metric.score,
            "Baseline_Eval_Validity_Score": eval_score,
        })

        time.sleep(25)

    df = pd.DataFrame(all_results)

    print("\n========================================")
    print("FULL BASELINE EVALUATION METRICS")
    print("========================================")
    print(f"Total Interviews Processed: {len(df)}")
    print(f"Avg Baseline Prompt Compliance: {df['Baseline_Prompt_Compliance_Score'].mean():.2f}")
    print(f"Avg Baseline Semantic Relevance: {df['Baseline_Semantic_Relevance_Score'].mean():.2f}")
    print(f"Avg Baseline Structural Consistency: {df['Baseline_Structural_Consistency_Score'].mean():.2f}")

    valid_scorecards = df['Baseline_Eval_Validity_Score'].dropna()
    if not valid_scorecards.empty:
        print(f"Avg Baseline Evaluation Validity: {valid_scorecards.mean():.2f} ({len(valid_scorecards)} scorecards evaluated)")
    else:
        print("Avg Baseline Evaluation Validity: N/A (No scorecards found)")

    csv_filename = "full_agent_evaluation_report_ZERO_SHOT_BASELINE.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nDetailed baseline report saved to: {csv_filename}")

if __name__ == "__main__":
    evaluate_all_interviews_baseline("/content/drive/MyDrive/Evaluation Dataset")

Output()

Starting BASELINE evaluation of 13 interviews...

Evaluating Baseline: Interview Eval #1.json...


Output()

Output()

--- Baseline Judgement for Interview Eval #1.json ---
Prompt Compliance: 0.0
Semantic Relevance: 0.2
Structural Consistency: 0.4
Evaluation Validity: N/A
----------------------------------------


...

Evaluating Baseline: Interview Eval #12.json...


Output()

Output()

Output()

--- Baseline Judgement for Interview Eval #12.json ---
Prompt Compliance: 0.0
Semantic Relevance: 0.2
Structural Consistency: 0.3
Evaluation Validity: 0.2
----------------------------------------

FULL BASELINE EVALUATION METRICS
Total Interviews Processed: 13
Avg Baseline Prompt Compliance: 0.00
Avg Baseline Semantic Relevance: 0.40
Avg Baseline Structural Consistency: 0.57
Avg Baseline Evaluation Validity: 0.19 (8 scorecards evaluated)

Detailed baseline report saved to: full_agent_evaluation_report_ZERO_SHOT_BASELINE.csv


# **Gemini**

In [ ]:
import json
import glob
import os
import time
import pandas as pd
from google.colab import userdata

from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import DeepEvalBaseLLM
from langchain_google_genai import ChatGoogleGenerativeAI

os.environ["GOOGLE_API_KEY"] = userdata.get('Gemini_API_Key')

class GoogleGemini(DeepEvalBaseLLM):
    def __init__(self, model_name="gemini-2.5-pro"):
        self.model_name = model_name
        self.model = ChatGoogleGenerativeAI(
            model=model_name,
            max_retries=3
        )

    def load_model(self):
        return self.model

    def generate(self, prompt: str) -> str:
        return self.model.invoke(prompt).content

    async def a_generate(self, prompt: str) -> str:
        res = await self.model.ainvoke(prompt)
        return res.content

    def get_model_name(self):
        return self.model_name


def load_interview_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    cv_string = json.dumps(data.get('candidate_cv', {}), indent=2)
    combined_input = f"JOB DESCRIPTION:\n{data.get('job_description', '')}\n\nCANDIDATE CV:\n{cv_string}"

    transcript_str = "\n".join([f"{t['role'].upper()}: {t['content']}" for t in data.get('transcript', [])])

    scorecard = data.get('scorecard', None)
    scorecard_str = json.dumps(scorecard, indent=2) if scorecard else None

    return combined_input, transcript_str, scorecard_str

def evaluate_all_interviews_baseline(folder_path):
    gemini_judge = GoogleGemini("gemini-2.5-pro")

    prompt_compliance_metric = GEval(
        name="Prompt-Compliance Baseline",
        criteria="Score the interviewer agent's prompt compliance and conversational behavior on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    semantic_relevance_metric = GEval(
        name="Semantic Relevance Baseline",
        criteria="Score the semantic alignment of the interviewer's questions with the Job Description and Candidate CV on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    structural_consistency_metric = GEval(
        name="Structural Consistency Baseline",
        criteria="Score the structural flow and logical progression of the interview on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    evaluation_validity_metric = GEval(
        name="Evaluation Validity Baseline",
        criteria="Score the logic of the generated scorecard on a scale of 1 to 10. Output a float converted to a 0.0-1.0 scale.",
        evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
        model=gemini_judge
    )

    all_results = []
    files = glob.glob(os.path.join(folder_path, "*.json"))

    print(f"Starting BASELINE evaluation of {len(files)} interviews...\n")

    for file_path in files:
        file_name = os.path.basename(file_path)
        print(f"Evaluating Baseline: {file_name}...")

        system_input, transcript_output, scorecard_output = load_interview_data(file_path)

        interviewer_test_case = LLMTestCase(
            input=system_input,
            actual_output=transcript_output
        )

        prompt_compliance_metric.measure(interviewer_test_case)
        semantic_relevance_metric.measure(interviewer_test_case)
        structural_consistency_metric.measure(interviewer_test_case)

        eval_score = None
        eval_reason = "No scorecard provided in JSON."

        if scorecard_output:
            scorecard_test_case = LLMTestCase(
                input=f"{system_input}\n\nTRANSCRIPT:\n{transcript_output}",
                actual_output=scorecard_output
            )
            evaluation_validity_metric.measure(scorecard_test_case)
            eval_score = evaluation_validity_metric.score
            eval_reason = evaluation_validity_metric.reason

        print(f"--- Baseline Judgement for {file_name} ---")
        print(f"Prompt Compliance: {prompt_compliance_metric.score}")
        print(f"Semantic Relevance: {semantic_relevance_metric.score}")
        print(f"Structural Consistency: {structural_consistency_metric.score}")
        print(f"Evaluation Validity: {eval_score if eval_score is not None else 'N/A'}")
        print("-" * 40)

        all_results.append({
            "Interview_ID": file_name,
            "Baseline_Prompt_Compliance_Score": prompt_compliance_metric.score,
            "Baseline_Semantic_Relevance_Score": semantic_relevance_metric.score,
            "Baseline_Structural_Consistency_Score": structural_consistency_metric.score,
            "Baseline_Eval_Validity_Score": eval_score,
        })

        time.sleep(25)

    df = pd.DataFrame(all_results)

    print("\n========================================")
    print("FULL BASELINE EVALUATION METRICS")
    print("========================================")
    print(f"Total Interviews Processed: {len(df)}")
    print(f"Avg Baseline Prompt Compliance: {df['Baseline_Prompt_Compliance_Score'].mean():.2f}")
    print(f"Avg Baseline Semantic Relevance: {df['Baseline_Semantic_Relevance_Score'].mean():.2f}")
    print(f"Avg Baseline Structural Consistency: {df['Baseline_Structural_Consistency_Score'].mean():.2f}")

    valid_scorecards = df['Baseline_Eval_Validity_Score'].dropna()
    if not valid_scorecards.empty:
        print(f"Avg Baseline Evaluation Validity: {valid_scorecards.mean():.2f} ({len(valid_scorecards)} scorecards evaluated)")
    else:
        print("Avg Baseline Evaluation Validity: N/A (No scorecards found)")

    # Saved with a distinct name referencing Gemini
    csv_filename = "full_agent_evaluation_report_gemini_ZERO_SHOT_BASELINE.csv"
    df.to_csv(csv_filename, index=False)
    print(f"\nDetailed baseline report saved to: {csv_filename}")

# Run the evaluation
if __name__ == "__main__":
    evaluate_all_interviews_baseline("/content/drive/MyDrive/Evaluation Dataset")

Output()

Starting BASELINE evaluation of 13 interviews...

Evaluating Baseline: Interview Eval #1.json...


Output()

Output()

--- Baseline Judgement for Interview Eval #1.json ---
Prompt Compliance: 0.5
Semantic Relevance: 0.2
Structural Consistency: 0.3
Evaluation Validity: N/A
----------------------------------------


...

Evaluating Baseline: Interview Eval #12.json...


Output()

Output()

Output()

--- Baseline Judgement for Interview Eval #12.json ---
Prompt Compliance: 0.6
Semantic Relevance: 0.5
Structural Consistency: 0.3
Evaluation Validity: 1.0
----------------------------------------

FULL BASELINE EVALUATION METRICS
Total Interviews Processed: 13
Avg Baseline Prompt Compliance: 0.58
Avg Baseline Semantic Relevance: 0.64
Avg Baseline Structural Consistency: 0.49
Avg Baseline Evaluation Validity: 0.57 (8 scorecards evaluated)

Detailed baseline report saved to: full_agent_evaluation_report_gemini_ZERO_SHOT_BASELINE.csv
